In [3]:
from langchain.tools import tool, ToolRuntime

In [4]:
@tool 
def read_email(runtime: ToolRuntime) -> str: 
    """Read an email from the given address.""" # take email from state 
    return runtime.state["email"]

In [5]:
@tool 
def send_email(body: str) -> str: 
    """Send an email to the given address with the given subject and body.""" # fake email sending 
    return f"Email sent"

In [6]:
from langchain.agents import create_agent, AgentState 
from langgraph.checkpoint.memory import InMemorySaver 
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langchain_ollama import ChatOllama

In [26]:
from dotenv import load_dotenv
import os

load_dotenv()


True

In [22]:
class EmailState(AgentState): 
    email: str 
agent = create_agent( 
    model="gpt-5-nano", 
    tools=[read_email, send_email], 
    state_schema=EmailState, 
    checkpointer=InMemorySaver(), 
    middleware=[ 
        HumanInTheLoopMiddleware( 
            interrupt_on={ 
                "read_email": False, 
                "send_email": True, 
                }, 
                description_prefix="Tool execution requires approval", 
                ), 
            ], 
        )

In [23]:
from langchain.messages import HumanMessage

In [24]:
config = {"configurable": {"thread_id": "1"}} 
response = agent.invoke( 
    {
         "messages": [HumanMessage(content="Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion."
         )
         ],
         "email": "Bonjour Sara, je vais être en retard pour notre réunion de demain. Pouvons-nous la reprogrammer ? Cordialement, Sofia" 
    }, 
         config=config 
        )

In [11]:
print(response)

{'messages': [HumanMessage(content='Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion.', additional_kwargs={}, response_metadata={}, id='6b8544f8-5373-4b30-a2ad-bb007ec4c905'), HumanMessage(content='Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion.', additional_kwargs={}, response_metadata={}, id='4056487e-c95a-4ac5-ad71-587118bde191'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 979, 'prompt_tokens': 200, 'total_tokens': 1179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dspe5aDp7ou9C8bZj2xJk8qFdXDbe

In [12]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Bonjour Sofia,\n\nMerci pour l'information. Pas de souci pour le retard.\n\nPour la reprogrammation de notre réunion de demain, quels créneaux vous conviennent ? Je suis disponible en fin de matinée ou en début d'après-midi. Dites-moi ce qui vous arrange et je confirmerai le nouveau rendez-vous.\n\nBien cordialement,\nSara"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Bonjour Sofia,\\n\\nMerci pour l\'information. Pas de souci pour le retard.\\n\\nPour la reprogrammation de notre réunion de demain, quels créneaux vous conviennent ? Je suis disponible en fin de matinée ou en début d\'après-midi. Dites-moi ce qui vous arrange et je confirmerai le nouveau rendez-vous.\\n\\nBien cordialement,\\nSara"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='5beee07644b1bd0ca54afa57761fee86')]


In [14]:
from langgraph.types import Command

In [15]:
response = agent.invoke( 
    Command( resume={"decisions": [{"type": "approve"}]} ), 
    config=config # Le même thread ID pour reprendre la conversation
     ) 
print(response['messages'][-1].content)

J’ai lu votre e-mail et j’ai envoyé la réponse suivante dans le même fil de discussion:

Bonjour Sofia,

Merci pour l'information. Pas de souci pour le retard.

Pour la reprogrammation de notre réunion de demain, quels créneaux vous conviennent ? Je suis disponible en fin de matinée ou en début d'après-midi. Dites-moi ce qui vous arrange et je confirmerai le nouveau rendez-vous.

Bien cordialement,
Sara

Souhaitez-vous que je fasse l’une des choses suivantes ?
- Adapter le ton (plus formel ou plus décontracté).
- Proposer des créneaux précis (par exemple demain à 11h00 ou 14h30) plutôt que des créneaux généraux.
- Ajouter une demande de confirmation du nouveau rendez-vous ou inclure une invitation de calendrier.
- Vérifier le fuseau horaire et ajuster si nécessaire.

Dites-moi ce que vous préférez et je propage le nouveau message immédiatement.


In [18]:
response = agent.invoke( 
    Command( 
        resume={ 
            "decisions": [ 
                { "type": "reject", # Une explication sur les raisons du rejet 
                "message": " J’annule notre rendez-vous." 
                } 
                ] 
                } 
                ), 
                config=config # Le même thread ID pour reprendre la conversation
) 
print(response)

{'messages': [HumanMessage(content='Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion.', additional_kwargs={}, response_metadata={}, id='6b8544f8-5373-4b30-a2ad-bb007ec4c905'), HumanMessage(content='Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion.', additional_kwargs={}, response_metadata={}, id='4056487e-c95a-4ac5-ad71-587118bde191'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 979, 'prompt_tokens': 200, 'total_tokens': 1179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dspe5aDp7ou9C8bZj2xJk8qFdXDbe

In [25]:
from langgraph.types import Command 
response = agent.invoke( 
    Command( 
        resume={ 
            "decisions": [ 
                { "type": "edit", 
                "edited_action": 
                { # Le nom du Tool. 
                "name": "send_email", # Les arguments à passer au tool. 
                "args": 
                {"body": "Je suis désolée mais je dois annuler notre rendez-vous je ne serais pas libre. Sara"}, 
                } } ] } ), 
                config=config # Le même thread ID pour reprendre la conversation 
                ) 
print(response['messages'][-1].content)

J'ai lu votre e-mail et j'ai envoyé une réponse pour vous. Voici le détail de l'action effectuée:

- E-mail lu:
  Bonjour Sara, je vais être en retard pour notre réunion de demain. Pouvons-nous la reprogrammer ? Cordialement, Sofia

- Réponse envoyée:
  Je suis désolée mais je dois annuler notre rendez-vous je ne serais pas libre. Sara

Souhaitez-vous que je propose une nouvelle heure ou une nouvelle date, ou préférez-vous un message plus formel ?
